In [ ]:
import numpy as np
from scipy.signal import welch

# https://chatgpt.com/c/69dfa307-fc9c-8384-b590-0d0a869feb48

def classify_ecg_noise_segment(
    x,
    fs=200.0,
    band_50=(48.0, 52.0),
    band_emg=(20.0, 80.0),
    band_motion=(0.1, 5.0),
    band_total=(0.1, 90.0),
    min_peak_prominence_ratio=3.0,
):
    """
    Classify dominant noise in one ECG segment as:
        - 'powerline_50hz'
        - 'emg_broadband'
        - 'motion_artifact'
        - 'mixed_or_unclear'

    Parameters
    ----------
    x : array-like
        1D ECG segment.
    fs : float
        Sampling frequency in Hz. Default 200.
    band_50 : tuple(float, float)
        Frequency band around 50 Hz.
    band_emg : tuple(float, float)
        EMG-like broadband band.
    band_motion : tuple(float, float)
        Low-frequency motion artifact band.
    band_total : tuple(float, float)
        Reference band for normalization.
    min_peak_prominence_ratio : float
        How much stronger 50 Hz peak density should be than EMG background
        to be considered narrowband power-line dominated.

    Returns
    -------
    result : dict
        {
            'label': str,
            'scores': {
                'powerline_50hz': float,
                'emg_broadband': float,
                'motion_artifact': float
            },
            'features': {...},
            'freqs': f,
            'psd': pxx
        }
    """
    x = np.asarray(x, dtype=float).ravel()

    if x.size < max(64, int(fs)):
        raise ValueError("Segment too short. Use at least ~1 second, preferably 5-10 seconds.")

    # Remove DC
    x = x - np.mean(x)

    # Welch PSD
    nperseg = min(len(x), max(128, int(fs * 2)))
    if nperseg % 2 == 1:
        nperseg -= 1
    if nperseg < 64:
        nperseg = min(len(x), 64)

    f, pxx = welch(
        x,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=nperseg // 2,
        detrend="constant",
        scaling="density",
    )

    eps = 1e-16

    def band_mask(band):
        lo, hi = band
        return (f >= lo) & (f <= hi)

    def band_power(band):
        m = band_mask(band)
        if not np.any(m):
            return 0.0
        return np.trapz(pxx[m], f[m])

    total_power = band_power(band_total) + eps
    power_50 = band_power(band_50)
    power_emg = band_power(band_emg)
    power_motion = band_power(band_motion)

    # Peak near 50 Hz
    m50 = band_mask(band_50)
    if np.any(m50):
        p50_max = np.max(pxx[m50])
        f50_peak = f[m50][np.argmax(pxx[m50])]
    else:
        p50_max = 0.0
        f50_peak = np.nan

    # EMG background excluding narrow 50 Hz neighborhood
    m_emg = band_mask(band_emg)
    m_exclude_50 = (f >= 47.0) & (f <= 53.0)
    m_emg_bg = m_emg & (~m_exclude_50)

    if np.any(m_emg_bg):
        emg_bg_mean = np.mean(pxx[m_emg_bg]) + eps
        emg_bg_std = np.std(pxx[m_emg_bg]) + eps
    else:
        emg_bg_mean = eps
        emg_bg_std = eps

    peak_prominence_ratio = p50_max / emg_bg_mean

    # Spectral flatness in EMG band:
    # broadband EMG tends to be flatter than narrowband interference
    if np.any(m_emg_bg):
        p_emg_bg = pxx[m_emg_bg] + eps
        spectral_flatness_emg = np.exp(np.mean(np.log(p_emg_bg))) / np.mean(p_emg_bg)
    else:
        spectral_flatness_emg = 0.0

    # Normalize band energies
    r50 = power_50 / total_power
    remg = power_emg / total_power
    rmotion = power_motion / total_power

    # Heuristic scores
    # 1) Power-line: strong narrow 50 Hz peak + meaningful 50 Hz energy
    score_powerline = r50 * min(peak_prominence_ratio / min_peak_prominence_ratio, 3.0)

    # 2) EMG: substantial high-frequency energy, relatively broadband, not dominated by sharp 50 Hz line
    score_emg = remg * (0.5 + spectral_flatness_emg) / max(1.0, peak_prominence_ratio / min_peak_prominence_ratio)

    # 3) Motion: substantial low-frequency energy
    score_motion = rmotion

    scores = {
        "powerline_50hz": float(score_powerline),
        "emg_broadband": float(score_emg),
        "motion_artifact": float(score_motion),
    }

    # Decision logic
    best_label = max(scores, key=scores.get)
    sorted_scores = sorted(scores.items(), key=lambda kv: kv[1], reverse=True)
    best_name, best_score = sorted_scores[0]
    second_name, second_score = sorted_scores[1]

    # Extra guards
    is_strong_50hz = (r50 > 0.08) and (peak_prominence_ratio >= min_peak_prominence_ratio)
    is_emg_like = (remg > 0.20) and (spectral_flatness_emg > 0.2) and not is_strong_50hz
    is_motion_like = (rmotion > 0.35)

    if is_strong_50hz and best_name == "powerline_50hz":
        label = "powerline_50hz"
    elif is_motion_like and best_name == "motion_artifact":
        label = "motion_artifact"
    elif is_emg_like and best_name == "emg_broadband":
        label = "emg_broadband"
    else:
        # If top two are close, call it mixed/unclear
        if best_score < 0.05 or (second_score > 0 and best_score / second_score < 1.3):
            label = "mixed_or_unclear"
        else:
            label = best_name

    result = {
        "label": label,
        "scores": scores,
        "features": {
            "relative_power_50hz": float(r50),
            "relative_power_emg": float(remg),
            "relative_power_motion": float(rmotion),
            "peak_50hz_frequency": float(f50_peak) if not np.isnan(f50_peak) else None,
            "peak_50hz_prominence_ratio": float(peak_prominence_ratio),
            "spectral_flatness_emg_band": float(spectral_flatness_emg),
            "total_power_ref_band": float(total_power),
        },
        "freqs": f,
        "psd": pxx,
    }
    return result